In [0]:
from pyspark.sql.functions import *

In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/inbound/emp_details.csv")

In [0]:
df.display()

# select command

In [0]:
df.select(df.empid,df.deptno).display()

In [0]:


df.select(col("empid"),col("deptno"),col("state"),col("mobilenumber")).display()

#  withColumn ((adding new column))

In [0]:


df.withColumn("Gender_desc", when (col("gender")=="M","Male").otherwise("Female")).display()


In [0]:
df.withColumn("fullname", concat(df.firstname,lit(" "),df.lastname)).display()

In [0]:
df.withColumn("bonus",df.salary*0.10).display()

In [0]:
df.withColumn("salary",col("salary").cast("int")).display()

# with column renamed

In [0]:
df.withColumnRenamed("deptno","department_number").display()

In [0]:
for i in df.columns:
    df.withColumnRenamed(i,"emp_"+i)
    
df.display()



#  DROP Command
### drop single column


In [0]:
df.drop(col("lastname")).display()
df.drop(df.age).display()

### drop multi columns

In [0]:
df.drop("firstname","lastname","age").display()

In [0]:
list=["phonenumber","state","mail"]

df.drop(*list).display()

# FILTER

In [0]:
df.filter(df.salary>2000).display()

In [0]:
df.filter(col("salary")>3000).display()

# MULTIFILTER

In [0]:
df.filter((df.salary > 25) & (df.gender == 'M')).display()

In [0]:
df.filter(  (col("deptno")==10)  & (col("gender")== "M" )).display()

# DROP DUPLICATES

In [0]:
lst = [
    (1, 'srinivas', 'male', 30000),
    (1, 'srinivas', 'male', 30000),
    (2, 'phani', 'male', 40000),
    (3, 'srinivas', 'male', 30000)
]

df1 = spark.createDataFrame(
    lst,
    ['id', 'name', 'gender', 'salary']
)

df1.display()

In [0]:
df1.dropDuplicates().display()

In [0]:
df1.dropDuplicates(["name"]).display()

In [0]:
df1.dropDuplicates(["name"]).display()

# Distinct

In [0]:
df1.distinct().display()

# order by 

In [0]:
df.orderBy(col("age")).display()

In [0]:
df.orderBy(df.salary).display()

In [0]:
df.orderBy(length(col("firstname"))).display() 
df.orderBy(length(df.firstname)).display()

df.orderBy(length(col("lastname"))).display()
df.orderBy(length(df.lastname)).display()

In [0]:
df.orderBy("state",col("salary").desc()).display()

# Sort

In [0]:
df.sort(df.age.desc()).display()

In [0]:
df.sort(col("age").asc()).display()

# GroupBy 
### we cant use only groupby we have to use any aggregate function like count , sum , min , max,avg


In [0]:
df.groupBy("deptno","gender").count().display()

In [0]:
df.groupBy("deptno").count().display()

In [0]:
df.groupBy("deptno").agg(sum("salary").alias("Total_salary")).display()

In [0]:
df.groupBy("deptno").agg(min("salary").alias("Minimum_salary")).display()

In [0]:
df.groupBy("deptno").agg(max("salary").alias("maximum_salary")).display()

In [0]:
df.groupBy("deptno").agg(avg("salary").alias("avgerage_salary")).display()

In [0]:
df.groupby("deptno").agg(sum("salary").alias("Total_salary"),min("salary").alias("Minimum_salary"),max("salary").alias("maximum_salary"),avg("salary").alias("avgerage_salary")).display()

# JOINS

In [0]:
e_df = spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/inbound/emp.csv")
e_df.display()


In [0]:
d_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/inbound/dept.csv")
d_df.display()

### inner join

In [0]:
e_df.join(d_df,e_df.deptno==d_df.deptid,"inner").display()

### full outer


In [0]:
e_df.join(d_df,e_df.deptno==d_df.deptid,"fullouter").display()

### left join

In [0]:
e_df.join(d_df,e_df.deptno==d_df.deptid,"left").display()

### leftsemi

In [0]:
e_df.join(d_df,e_df.deptno==d_df.deptid,"leftsemi").display()

### leftanti
Returns only the rows from the left DataFrame that have **no match** in the right DataFrame (only left-side columns are kept). It is the opposite of leftsemi.

In [0]:
e_df.join(d_df,e_df.deptno==d_df.deptid,"leftanti").display()

### right join 

In [0]:
e_df.join(d_df,e_df.deptno==d_df.deptid,"right").display()

## rightanti and rightsemi
### right anti join is not available in pyspark for that we have to change df using leftsemi and left anti

In [0]:
d_df.join(e_df,e_df.deptno==d_df.deptid,"leftanti").display()

In [0]:
d_df.join(e_df,e_df.deptno==d_df.deptid,"leftsemi").display()

### collect_list

In [0]:
data = [
    ("A", "apple"),
    ("A", "banana"),
    ("B", "orange"),
    ("B", "grape"),
    ("C", "kiwi")
]

df = spark.createDataFrame(data, ["cust_id", "fruit"])

display(df)

In [0]:
df.groupBy("cust_id").agg(collect_list("fruit").alias("fruit_list")).display()

# windowsfunction

In [0]:
from pyspark.sql import *            # SparkSession, DataFrame, Row, Column, Window, etc.
from pyspark.sql.functions import *  # col, sum, max, dense_rank, rank, row_number, collect_list, etc.
from pyspark.sql.types import *      # StructType, StructField, StringType, IntegerType, etc.
from pyspark.sql.window import Window

In [0]:
%sql
create or replace table shivasai_catalog.bronze.windowsfunctions
(
    empid int,
    emp_name string,
    salary int,
    gender varchar(2),
    deptno int
)

In [0]:
%sql
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (101, 'Prasad', 25000, 'M', 10);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (102, 'Rahul', 30000, 'M', 20);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (103, 'aditya', 40000, 'M', 30);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (104, 'Pavani', 20000, 'F', 40);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (105, 'anupama', 35000, 'F', 50);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (106, 'viswas', 25000, 'M', 10);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (107, 'akhila', 45000, 'F', 20);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (108, 'prajol', 35000, 'M', 30);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (109, 'venu', 40000, 'M', 40);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (110, 'Sujatha', 20000, 'F', 50);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (111, 'Krishna', 40000, 'M', 10);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (112, 'ahmad', 20000, 'M', 20);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (113, 'Anusha', 30000, 'F', 30);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (114, 'prasanth', 40000, 'M', 40);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (115, 'aravind', 20000, 'M', 50);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (116, 'kishore', 25000, 'M', 10);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (117, 'Sravani', 30000, 'F', 20);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (118, 'siva krishna', 40000, 'M', 30);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (119, 'satish', 20000, 'M', 40);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (120, 'Sunitha', 25000, 'F', 50);
INSERT INTO shivasai_catalog.bronze.windowsfunctions VALUES (121, 'rajendra', 50000, 'M', 10);

In [0]:
%sql
select w.*, 
dense_rank() over(order by w.salary DESC) as dense_rank,
rank() over(order by w.salary DESC) as rank,
row_number() over(order by w.salary DESC) as row_number
from shivasai_catalog.bronze.windowsfunctions w

In [0]:
df=spark.read.table("shivasai_catalog.bronze.windowsfunctions")
df.display()

In [0]:
from pyspark.sql.window import Window

# dense rank

In [0]:

df.withColumn("dense_rank",dense_rank().over(Window.orderBy(df.salary.desc()))).display()

# rank

In [0]:
df.withColumn("rank",rank().over(Window.orderBy(df.salary.desc()))).display()

# row_number

In [0]:
df.withColumn("row_number",row_number().over(Window.orderBy(df.salary.desc()))).display()

# partition by deptno order by salary with dense rank function

In [0]:
df.withColumn("partition_dense_rank",dense_rank().over(Window.partitionBy(df.deptno).orderBy(df.salary.desc()))).display()

# partition by deptno order by salary with rank function

In [0]:
df.withColumn("partition_rank",rank().over(Window.partitionBy(df.deptno).orderBy(df.salary.desc()))).display()

# partition by deptno order by salary with row function

In [0]:
df.withColumn("partition_row_number",row_number().over(Window.partitionBy(df.deptno).orderBy(df.salary.desc()))).display()

### each dep wise total salary

In [0]:
df.groupBy("deptno").agg(sum("salary").alias("total_salary")).display()

### Q)2nd highest total salry dept wise

In [0]:
dept_df=df.groupBy("deptno").agg(sum("salary").alias("total_salary"))

dept_df=dept_df.withColumn("dense_rank",dense_rank().over(Window.orderBy(col("total_salary").desc())))

dept_df.filter(dept_df.dense_rank==2).display()


### Q)dept wise 2nd highst salary

In [0]:
df=spark.read.table("shivasai_catalog.bronze.windowsfunctions")

df=df.withColumn(
    "dense_rank",dense_rank().over(
        Window.partitionBy(df.deptno)
            .orderBy(df.salary.desc())))


df.filter(df.dense_rank==2).display()


# LEAD AND LAG

In [0]:
%sql
create or replace table shivasai_catalog.bronze.sales
(
      year INT,
    quarter INT,
    amount INT
)

In [0]:
%sql

INSERT INTO shivasai_catalog.bronze.sales VALUES (2018, 1, 4000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2018, 2, 3000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2018, 3, 2000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2018, 4, 8000);

INSERT INTO shivasai_catalog.bronze.sales VALUES (2019, 1, 4500);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2019, 2, 9000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2019, 3, 8000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2019, 4, 4000);

INSERT INTO shivasai_catalog.bronze.sales VALUES (2020, 1, 6000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2020, 2, 2000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2020, 3, 4000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2020, 4, 8000);

INSERT INTO shivasai_catalog.bronze.sales VALUES (2021, 1, 4500);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2021, 2, 3000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2021, 3, 2000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2021, 4, 8800);

INSERT INTO shivasai_catalog.bronze.sales VALUES (2022, 1, 4500);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2022, 2, 3000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2022, 3, 8000);
INSERT INTO shivasai_catalog.bronze.sales VALUES (2022, 4, 4500);


In [0]:
%sql
select
 *,
 lag(amount) over ( order by year, quarter) as lag_amount,
lag(amount,2) over ( order by year, quarter) as lag_amount_2,
 lag(amount,1,0) over ( partition by year order by year, quarter) as lag_partition_1_no,
lag(amount,2) over ( partition by year order by year, quarter) as lag_partition_2_no,
lead(amount) over(order by year,quarter) as lead_amount,
lead(amount,2) over(order by year,quarter) as lead_2_amount,
lead(amount,1,0) over(partition by year order by year,quarter) as lead_partition_1_no,
lead(amount,2,0) over(partition by year order by year,quarter) as lead_partition_2_no
from shivasai_catalog.bronze.sales
order by year,quarter

In [0]:
from pyspark.sql.functions import lag, lead, col
from pyspark.sql.window import Window

# Read the existing table into a DataFrame
df = spark.table("shivasai_catalog.bronze.sales")

# Window without partition
w = Window.orderBy("year", "quarter")

# Window partitioned by year
w_year = Window.partitionBy("year").orderBy("year", "quarter")

# Apply lag and lead functions
df_result = (
    df
    .withColumn("lag_amount", lag("amount").over(w))
    .withColumn("lag_amount_2", lag("amount", 2).over(w))
    .withColumn("lag_partition_1_no", lag("amount", 1, 0).over(w_year))
    .withColumn("lag_partition_2_no", lag("amount", 2).over(w_year))
    .withColumn("lead_amount", lead("amount").over(w))
    .withColumn("lead_2_amount", lead("amount", 2).over(w))
    .withColumn("lead_partition_1_no", lead("amount", 1, 0).over(w_year))
    .withColumn("lead_partition_2_no", lead("amount", 2, 0).over(w_year))
)

display(df_result.orderBy("year", "quarter"))      

# pivot and unpivot

In [0]:
data = [
    (1, "Maths", 85),
    (1, "Science", 90),
    (1, "English", 78),
    (2, "Maths", 88),
    (2, "Science", 76),
    (3, "Maths", 92),
    (3, "English", 81)
]

columns = ["student_id", "subject", "marks"]

df = spark.createDataFrame(data, columns)

df.display()

### pivot

In [0]:
from pyspark.sql.functions import *

pivot_df = df.groupBy("student_id").pivot("subject").agg(
    sum("marks")
)

display(pivot_df)

In [0]:
data = [
    (101, "2024-12-01", "UPI", 1500),
    (102, "2024-12-01", "CARD", 2200),
    (103, "2024-12-01", "COD", 800),
    (104, "2024-12-02", "UPI", 1800),
    (105, "2024-12-02", "CARD", 2500),
    (106, "2024-12-02", "UPI", 700),
    (107, "2024-12-03", "COD", 1200),
    (108, "2024-12-03", "CARD", 3000)
]

columns = ["order_id", "order_date", "payment_mode", "amount"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
df.groupBy("order_date").pivot("payment_mode").agg(sum("amount")).display()

# stack

In [0]:
pivot_df = df.groupBy("order_date").pivot("payment_mode").agg(sum("amount"))

unpivot_df = pivot_df.select(
    "order_date",
    expr("""
        stack(3,
            'UPI_1', UPI,
            'CARD_1', CARD,
            'COD_1', COD
        ) as (payment_mode, amount)
    """)
)
unpivot_df.display()